# 05 · Scaling Laws Analysis
Standard Kaplan scaling and the DLCM compression-aware extension L(N, D, R, P).

**Papers:** Kaplan 2020 ([2001.08361](https://arxiv.org/abs/2001.08361)), DLCM ([2512.24617](https://arxiv.org/abs/2512.24617)) Section 6

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## 1. Standard Kaplan scaling law: L(N) = (Nc/N)^alpha

In [ ]:
def kaplan(N, Nc, alpha):
    return (Nc / N) ** alpha

N_pts = np.array([1e6, 1e7, 1e8, 1e9, 1e10])
L_pts = np.array([3.80, 3.10, 2.50, 2.00, 1.65])

popt, _ = curve_fit(kaplan, N_pts, L_pts, p0=[1e13, 0.076])
Nc, alpha = popt
N_range = np.logspace(6, 11, 200)

plt.figure(figsize=(8, 5))
plt.loglog(N_pts, L_pts, 'o', color='#C00000', ms=8, label='Empirical (schematic)')
plt.loglog(N_range, kaplan(N_range, Nc, alpha), '-', color='#2E75B6',
           label=f'Fit: alpha={alpha:.3f}')
plt.xlabel('Parameters N'); plt.ylabel('Loss L')
plt.title('Kaplan Scaling Law: L(N) = (Nc/N)^alpha')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"Nc = {Nc:.2e},  alpha = {alpha:.4f}")

## 2. DLCM compression-aware effective parameters

In [ ]:
def N_eff(N, R, P):
    """Backbone (P*N params) operates on M=L/R positions => R-times more efficient per position."""
    return P * N * R + (1 - P) * N

N_total = 1e9
P = 0.7
print(f"Total params: {N_total:.0e} | Backbone fraction P={P}")
print()
for R in [1, 2, 4, 8]:
    ne = N_eff(N_total, R, P)
    print(f"  R={R}: N_eff={ne:.2e}  (x{ne/N_total:.2f} effective utilization)")

## 3. FLOP analysis: DLCM vs. standard LLM at matched inference budget

In [ ]:
def flops_transformer(L, d, n_layers):
    return 2 * n_layers * L * (4 * d**2 + 2 * L * d)

def flops_dlcm(L, d_tok, d_con, n_tok, n_con, R):
    M = L // R
    return (flops_transformer(L, d_tok, n_tok)
          + flops_transformer(M, d_con, n_con)
          + 2 * L * M * d_tok)

L, d, n = 1024, 512, 12
f_std = flops_transformer(L, d, n)
print(f"Standard LLM: {f_std:.3e} FLOPs  (L={L}, d={d}, layers={n})")
print()
for R in [2, 4, 8]:
    d_con  = int(d * (1 + (R-1) * 0.3))
    f_dlcm = flops_dlcm(L, d//2, d_con, 4, n, R)
    savings = (f_std - f_dlcm) / f_std * 100
    print(f"  R={R}: DLCM FLOPs={f_dlcm:.3e}  savings={savings:.1f}%  d_concept={d_con}")

## 4. Decoupled muP: learning rate scaling (DLCM Eq. 19-20)

In [ ]:
d_base, d_token, d_concept = 256, 512, 1024
lr_base = 1e-3

lr_token   = lr_base * (d_base / d_token)
lr_concept = lr_base * (d_base / d_concept)

print("Decoupled muP learning rates:")
print(f"  lr_token   (d={d_token}):   {lr_token:.2e}")
print(f"  lr_concept (d={d_concept}): {lr_concept:.2e}")
print()
print("Without decoupled muP: wider concept backbone trains with oversized LR => instability.")
print("Decoupling enables zero-shot hyperparameter transfer from proxy to full scale.")